# Sparse-scaling benchmark: Pt

Bulk fcc Pt (example05) with spin-orbit coupling from the DFT run, `nawf = 18` per primitive cell
(the ladder runs to d = 4, `nawf = 288`, for now). Pt is a cubic metal with inversion and time reversal. It tests the
spin and orbital Hall conductivities, which are the first properties of the benchmark that use
the doubled spin and orbital operators `Sj`/`Lj`. It also runs every Si property that has a
Fermi surface here, on the bottom-of-the-spectrum window and on an interior window through E_F.

A cell doubled `d` times is still the same crystal, so every property per primitive cell must
look the same at every size. Each figure shows one sparse pipeline. The colored lines are its
sizes, and the dashed black line is the dense reference on the primitive cell (`reference_d00`).
The curves come from [observables.py](../../observables.py), which normalizes them per primitive
cell, and the figures from [plots.py](../../plots.py). The settings are in
[system.py](system.py). A pipeline that does not run a property has no panel for it.

**Before the first ladder** there are no measured notes here yet. The expectations below come
from a smoke test at a 12³ mesh, d ≤ 1 (2026-10-26):

- With the exact H(R), sparse d = 0 matches the dense reference to 3×10⁻¹⁰ in both spin Hall routes and the orbital Hall conductivity, 4×10⁻¹⁴ in the Kubo σ, exactly in the Boltzmann σ, 10⁻⁹ in ε″, 10⁻³ in the DoS and 3×10⁻³ in the spin circular dichroism; the bands differ by 2 meV RMS. The band-curvature family is the exception: Hall 1–2%, Nernst 5–7%, `inv_mass_dos` 0.25 and `inv_mass_trace` 0.08, at d = 0 already, so a difference between the sparse and dense curvature kernels, not doubling.
- d = 1 matches d = 0 to 10⁻¹² in every curve, the spin and orbital Hall conductivities and the band curvature included (measured with a 23-shell cutoff before the switch to the exact H(R)), except σ^z_xz of the spin circular dichroism, which is 7×10⁻⁴ of its own (near-zero) size.

In [ ]:
import os
import sys

# the shared modules live two levels up, in benchmarks/sparse_scaling
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))
from plots import Benchmark

bench = Benchmark('pt', root=os.path.abspath('runs'), symlog=False)
show = bench.show

## 1. Density of states

The DoS per primitive cell. The PDOS is not compared: with spin-orbit coupling, both spinor copies
of an orbital share a file prefix, so the undoubled cell writes the second over the first.

In [ ]:
show(['dos'], ncol=1)

## 2. Band structure at d = 0

Doubling folds the band path, so only the undoubled cell is compared: the dense reference against
`sparse` d = 0, on PAOFLOW's default fcc path. The sparse path holds the
well-projected bands, and NaN above the usable top (the PAO shift minus 0.2 eV). The right panel
is the difference per band within ±3 eV of E_F. `collect.py` reports the RMS over every band as
`dev_bands_vs_ref`, in eV.

Measured with the band path alone (2026-10-26): with the exact H(R) the sparse bands lie within
3.4 meV of the dense ones within 1 eV of E_F (0.6 meV RMS) and 20 meV anywhere, which is the off-
grid interpolation convention. The largest shell cutoff the QE grid allows (23 shells) would move
them by up to 47 meV near E_F and 1.1 eV elsewhere, which is why Pt runs without one.

In [ ]:
bench.show_bands()

## 3. Boltzmann transport

Constant relaxation time, at 300 K, against the chemical potential. The Hall tensor trace needs
the band curvature, so only `sparse_curvature` has it.

In [ ]:
show(['sigma_xx', 'sigma_zz', 'seebeck_xx', 'kappa_xx', 'pf_xx', 'hall_trace'])

## 4. Hall and Nernst tensors

The Hall components with three different indices, and two Nernst components. Both need the band
curvature, so only `sparse_curvature` has them.

In [ ]:
show(['hall_xyz', 'hall_xzy', 'hall_yzx', 'nernst_xyz', 'nernst_yxz'])

## 5. Band-diagonal and Kubo conductivity

`conductivity` sums v_i v_j over states with adaptive smearing; `linear_response(response='cond')`
is the Kubo formula with a Lorentzian broadening of 0.05 eV, which sums over every pair of
states, so only `sparse_curvature` runs it.

In [ ]:
show(['cond_xx', 'cond_zz', 'lr_cond_xx', 'lr_cond_zz'], ncol=4)

## 6. Spin Hall conductivity

σ^z_xy and σ^x_yz, which cubic symmetry makes equal, by the Berry-curvature route (`spin_Hall`)
and by the Kubo route (`linear_response(response='shc')`, 0.05 eV broadening). Both sum over
every pair of states, so only `sparse_curvature` has them. σ^z_xz, which cubic symmetry zeroes,
is in the last section.

In [ ]:
show(['berry_shc_z_xy', 'berry_shc_x_yz', 'lr_shc_even_z_xy', 'lr_shc_even_x_yz'], ncol=2)

## 7. Spin circular dichroism

The frequency-dependent spin Hall conductivity of `spin_Hall(do_ac=True)`.

In [ ]:
show(['scd_re_z_xy', 'scd_im_z_xy'], ncol=2)

## 8. Orbital Hall conductivity

σ^z_xy and σ^x_yz of `orbital_Hall`, built on the orbital angular momentum operator `Lj`.

In [ ]:
show(['berry_ohc_z_xy', 'berry_ohc_x_yz'], ncol=2)

## 9. Optical response

Kubo-Greenwood ε″ (interband only: the Drude term of a metal is not in the sum). The grey line
is the window top minus 0.5 eV: above it a window run misses the transitions into states above
the window, and ε′ and the curves built from it need transitions at all energies.

In [ ]:
show(['epsi_xx', 'epsi_zz', 'epsr_xx', 'eels_xx', 'refl_xx', 'sigmar_xx'])

## 10. Effective masses

Smeared densities per k-point of the primitive cell, of the inverse DoS mass and of the mean
inverse principal mass (see the Si notebook for why only these symmetric functions compare).

In [ ]:
show(['inv_mass_dos', 'inv_mass_trace'], ncol=2)

## 11. Components that must vanish

Pt is cubic, so every off-diagonal Boltzmann and conductivity component vanishes, and so do
σ^z_xz of the spin and orbital Hall conductivities. Each panel divides the curve by the largest
value of its symmetry-allowed partner (the norm `collect.py` also uses for its deviation).

In [ ]:
show(
    [
        'sigma_xy',
        'sigma_xz',
        'sigma_yz',
        'seebeck_xy',
        'kappa_xy',
        'pf_xy',
        'hall_zero',
        'nernst_zero',
        'cond_xy',
        'lr_cond_xy',
        'berry_shc_z_xz',
        'berry_ohc_z_xz',
        'lr_shc_even_z_xz',
    ],
    normalized=True,
)

## 12. What is not tested here

- **The anomalous Hall conductivity, the Rashba-Edelstein response and the spin texture.** Pt has
  inversion and time reversal, so the first two vanish and every band is a Kramers pair whose
  spin has no fixed direction. Fe and Te test them.
- **`doping`**: Pt is a metal.
- **PDOS**: see section 1.